# 🛡️ Train AI Guardrail (Gemma 2B) on T4 GPU
This notebook fine-tunes Google's Gemma-2B-it using 4-bit QLoRA to act as a Brand-Safety Checker.

In [3]:
!pip install -q -U transformers peft bitsandbytes trl datasets accelerate huggingface_hub

In [5]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Gemma requires you to accept their license on HuggingFace first.
from huggingface_hub import notebook_login
notebook_login()

ModuleNotFoundError: Could not import module 'TrainingArguments'. Are this object's requirements defined correctly?

In [ ]:
# Load the dataset we generated
dataset = load_dataset('json', data_files='security_dataset.json', split='train')

def format_instruction(example):
    return f"Instruction:\n{example['instruction']}\n\nInput:\n{example['input']}\n\nOutput:\n{example['output']}"

# We don't need to map yet, SFTTrainer handles formatting

In [ ]:
# 4-Bit Quantization Configuration (Squeezes model into T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

In [ ]:
# LoRA Configuration (We only train ~1% of the model's brain to save time)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

In [ ]:
# Training Arguments optimized for 16GB T4 GPU
training_args = TrainingArguments(
    output_dir="./gemma-guardrail-results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=20,
    logging_steps=5,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    max_grad_norm=0.3,
    max_steps=100,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=512,
    dataset_text_field="input", # we use a formatting func usually, but trl supports custom
    formatting_func=lambda x: [format_instruction(item) for item in zip(x['instruction'], x['input'], x['output'])] if isinstance(x['input'], list) else [format_instruction(x)],
    args=training_args,
)

# Start Training!
trainer.train()

In [ ]:
# Save the fine-tuned adapters
trainer.model.save_pretrained("gemma-guardrail-adapter")
tokenizer.save_pretrained("gemma-guardrail-adapter")
print("Model saved! You can now download the 'gemma-guardrail-adapter' folder.")